In [1]:
import os
import sys
import requests
import pickle as pkl
from tqdm import tqdm
import sys
import time
import json
import requests
import itertools
import pandas as pd

import open_alex_utils
import importlib
importlib.reload(open_alex_utils)

import networkx as nx


In [2]:
BASEPMIDOPEN = '../data/pmid_to_openAlex'
PMIDS = '../data/pmids.pkl'
with open(PMIDS, 'rb') as f:
    pmids = pkl.load(f)
    
pmids_total = []

for study_design in pmids:
    for neg_pos in pmids[study_design]:
        for split in pmids[study_design][neg_pos]:
            pmids_total.extend(pmids[study_design][neg_pos][split])
            
            
pos_train_open_alex_ids = []
m = len(pmids_total)//50



pmid_oalex = {}

In [3]:
for n in tqdm(range(m)):
    pmid_str = '|'.join(map(str, pmids_total[50*n:50*(n+1)]))
    save_path = os.path.join(BASEPMIDOPEN, f'pmid_{n}.json')
    if not os.path.exists(save_path):
                print('file', n)
                response = requests.get(f"https://api.openalex.org/works?filter=pmid:{pmid_str}&per-page=200")
                while  response.status_code != 200:
                    response = requests.get(f"https://api.openalex.org/works?filter=pmid:{pmid_str}&per-page=200")
                for result in response.json()['results']:
                        pmid = int(result['ids']['pmid'].split('/')[-1])
                        pmid_oalex[pmid] = result['id']
                print('saving', n)
                with open(save_path, 'w') as f:
                        json.dump(pmid_oalex, f)

100%|████████████████████████████████| 606369/606369 [00:10<00:00, 57902.31it/s]


In [4]:
pmid_str = '|'.join(map(str, pmids_total[50*m:]))
save_path = os.path.join(BASEPMIDOPEN, f'pmid_{m}.json')
response = requests.get(f"https://api.openalex.org/works?filter=pmid:{pmid_str}&per-page=200")
while  response.status_code != 200:
                response = requests.get(f"https://api.openalex.org/works?filter=pmid:{pmid_str}&per-page=200")
for result in response.json()['results']:
                        pmid = int(result['ids']['pmid'].split('/')[-1])
                        pmid_oalex[pmid] = result['id']
with open(save_path, 'w') as f:
                        json.dump(pmid_oalex, f)

In [5]:
files = os.listdir(BASEPMIDOPEN)

In [6]:
pmid_to_oalexid = {}

In [7]:
for file in tqdm(files):
    if 'json' in file:
        file_path = os.path.join(BASEPMIDOPEN, file)
        try:
            with open(file_path) as f:
                conv_dict = json.load(f)
                pmid_to_oalexid.update(conv_dict)
        except:
            print(file)

100%|████████████████████████████████| 606371/606371 [00:59<00:00, 10258.83it/s]


In [8]:
pmid_to_oalexid = {int(k):v for k,v in pmid_to_oalexid.items()}

In [9]:
PMID_OALEXID_JSON = '../data/pmid_to_openAlex.json'

with open(PMID_OALEXID_JSON, 'w') as f:
    json.dump(pmid_to_oalexid, f)

In [10]:
with open(PMID_OALEXID_JSON, 'r') as f:
    test = json.load(f)

In [11]:
test

{'9154849': 'https://openalex.org/W1999172455',
 '17397918': 'https://openalex.org/W2066928792',
 '15187238': 'https://openalex.org/W2133314499',
 '2532365': 'https://openalex.org/W2009331888',
 '11673161': 'https://openalex.org/W2150498921',
 '18816090': 'https://openalex.org/W2080482257',
 '20643598': 'https://openalex.org/W2109913859',
 '31278649': 'https://openalex.org/W2956154093',
 '11819190': 'https://openalex.org/W2064727502',
 '9789195': 'https://openalex.org/W2013190052',
 '7398098': 'https://openalex.org/W1964627173',
 '6448702': 'https://openalex.org/W1977598014',
 '22907273': 'https://openalex.org/W1996581602',
 '14995619': 'https://openalex.org/W2100725564',
 '29423101': 'https://openalex.org/W2776678223',
 '20628338': 'https://openalex.org/W2084482022',
 '843131': 'https://openalex.org/W2028756159',
 '22360742': 'https://openalex.org/W2154940420',
 '8755888': 'https://openalex.org/W2098701903',
 '27357404': 'https://openalex.org/W2474107391',
 '12096937': 'https://openal

In [12]:
oalex_graph_dict = {}
src_dir = 'open_alex_citations'

for pmid, oalexid in tqdm(pmid_to_oalexid.items()):
    file_name = 'open_alex_pubmed_pmid' + str(pmid) + '.json'
    path = os.path.join(src_dir, file_name)
    if os.path.exists(path):
        if os.path.getsize(path) == 0:
            continue
        else:
            with open(path, 'r') as f:
                citations = json.load(f)
    oalex_graph_dict[oalexid] = citations

100%|██████████████████████████████| 3568321/3568321 [05:10<00:00, 11476.71it/s]


In [14]:
with open('../data/open_alex_graph.json', 'w') as f:
    json.dump(oalex_graph_dict, f)